# Supermarket Sales Forecasting — Preprocessing Pipeline
## Phase 3: Data Preprocessing
**Author:** R M Hamdhan  
**Batch:** HNDSE 25.1F | Kandy NIBM  
**Input:** supermarket_sales_cleaned.csv (output from Phase 2)  
**Output:** X_train, X_test, y_train, y_test (ready for ML models)

In [1]:
# ── IMPORTS ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import warnings
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 5)

print("✓ Libraries loaded")

✓ Libraries loaded


## 1. Load Cleaned Dataset
Loading the cleaned dataset saved from Phase 2 EDA notebook

In [2]:
# Load the cleaned dataset from Phase 2
df = pd.read_csv('../data/supermarket_sales_cleaned.csv')

# Convert Date back to datetime (CSV doesn't preserve datetime type)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)

print("✓ Dataset loaded successfully")
print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"\nColumns available:")
print(df.columns.tolist())

✓ Dataset loaded successfully
  Shape: 1000 rows × 24 columns
  Missing values: 0
  Date range: 2019-01-01 → 2019-12-31

Columns available:
['Invoice ID', 'Branch', 'CustomerID', 'City', 'Customer type', 'Gender', 'Product line', 'Unit price', 'Quantity', 'Tax 5%', 'Total', 'Date', 'Time', 'Payment', 'cogs', 'gross margin percentage', 'gross income', 'Rating', 'Longitude', 'Latitude', 'day_of_week', 'month', 'week', 'hour']


## 2. Drop Unnecessary Columns
Removing columns that are identifiers, redundant, or constant

In [3]:
# Columns to drop and the reason for each
columns_to_drop = {
    'Invoice ID'              : 'Unique identifier — no predictive value',
    'CustomerID'              : 'Unique identifier — no predictive value',
    'Longitude'               : 'Redundant — perfectly correlated with Branch/City',
    'Latitude'                : 'Redundant — perfectly correlated with Branch/City',
    'gross margin percentage' : 'Constant value (4.76%) across all rows — zero variance',
    'cogs'                    : 'Derived from Total — causes data leakage',
    'Tax 5%'                  : 'Derived from gross income — causes data leakage',
    'City'                    : 'Redundant — perfectly mapped to Branch',
    'Date'                    : 'Raw date replaced by engineered features',
    'Time'                    : 'Raw time replaced by hour feature',
    'week'                    : 'Too granular — month captures seasonality better',
    'month'                   : 'Will re-encode as number below',
    'day_of_week'             : 'Will re-encode as number below',
}

# Print what we're dropping and why
print("Dropping the following columns:")
for col, reason in columns_to_drop.items():
    print(f"  ✗ {col:30s} → {reason}")

# Actually drop them
df = df.drop(columns=list(columns_to_drop.keys()))

print(f"\n✓ Columns remaining: {len(df.columns)}")
print(f"  {df.columns.tolist()}")

Dropping the following columns:
  ✗ Invoice ID                     → Unique identifier — no predictive value
  ✗ CustomerID                     → Unique identifier — no predictive value
  ✗ Longitude                      → Redundant — perfectly correlated with Branch/City
  ✗ Latitude                       → Redundant — perfectly correlated with Branch/City
  ✗ gross margin percentage        → Constant value (4.76%) across all rows — zero variance
  ✗ cogs                           → Derived from Total — causes data leakage
  ✗ Tax 5%                         → Derived from gross income — causes data leakage
  ✗ City                           → Redundant — perfectly mapped to Branch
  ✗ Date                           → Raw date replaced by engineered features
  ✗ Time                           → Raw time replaced by hour feature
  ✗ week                           → Too granular — month captures seasonality better
  ✗ month                          → Will re-encode as number below
  ✗ da


## 3. Feature Engineering
Creating new meaningful features from existing columns

In [4]:
# ── FEATURE ENGINEERING ──────────────────────────────────

# 1. is_weekend — binary flag (1=weekend, 0=weekday)
# We need to reload date info from cleaned CSV for this
df_dates = pd.read_csv('../data/supermarket_sales_cleaned.csv')
df_dates['Date'] = pd.to_datetime(df_dates['Date'], dayfirst=True)

df['is_weekend'] = df_dates['Date'].dt.dayofweek.isin([5, 6]).astype(int)
df['day_num']    = df_dates['Date'].dt.dayofweek        # 0=Mon, 6=Sun
df['month_num']  = df_dates['Date'].dt.month            # 1=Jan, 12=Dec

# 2. price_per_item — unit price × quantity interaction
df['price_quantity'] = df['Unit price'] * df['Quantity']

print("✓ New features created:")
print(f"  is_weekend    : {df['is_weekend'].value_counts().to_dict()}")
print(f"  day_num range : {df['day_num'].min()} to {df['day_num'].max()}")
print(f"  month_num range: {df['month_num'].min()} to {df['month_num'].max()}")
print(f"  price_quantity sample: {df['price_quantity'].head(3).tolist()}")
print(f"\nShape after feature engineering: {df.shape}")

✓ New features created:
  is_weekend    : {0: 729, 1: 271}
  day_num range : 0 to 6
  month_num range: 1 to 12
  price_quantity sample: [746.9, 91.67999999999999, 324.31]

Shape after feature engineering: (1000, 15)


## 4. Encoding Categorical Variables
Converting text categories into numbers for ML models

In [5]:
# ── LABEL ENCODING ───────────────────────────────────────
# LabelEncoder converts text to numbers
# e.g. Branch: A→0, B→1, C→2

encoders = {}  # dictionary to store all encoders for later use

categorical_cols = ['Branch', 'Customer type', 'Gender', 
                    'Product line', 'Payment']

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"✓ {col:15s} encoded | Classes: {list(le.classes_)}")

print(f"\nDataset after encoding:")
print(df.head(3))
print(f"\nShape: {df.shape}")

✓ Branch          encoded | Classes: ['A', 'B', 'C']
✓ Customer type   encoded | Classes: ['Member', 'Normal']
✓ Gender          encoded | Classes: ['Female', 'Male']
✓ Product line    encoded | Classes: ['Electronic accessories', 'Fashion accessories', 'Food and beverages', 'Health and beauty', 'Home and lifestyle', 'Sports and travel']
✓ Payment         encoded | Classes: ['Cash', 'Credit card', 'Ewallet']

Dataset after encoding:
   Branch  Customer type  Gender  Product line  Unit price  Quantity   Total  \
0       0              0       0             3       74.69        10  746.90   
1       2              1       0             3       15.28         6   91.68   
2       0              1       1             3       46.33         7  324.31   

   Payment  gross income  Rating  hour  is_weekend  day_num  month_num  \
0        2     35.566667     9.1    13           0        3          2   
1        0     15.280000    10.0    10           0        0          5   
2        1     15.44

## 5. Define Features and Target Variable
Separating input features (X) from the target variable (y)

In [6]:
# ── DEFINE X AND y ───────────────────────────────────────

# Target variable — what we want to predict
y = df['gross income']

# Feature matrix — everything the model uses to make predictions
X = df.drop(columns=['gross income'])

print("✓ Features and target defined")
print(f"\nTarget variable (y): gross income")
print(f"  Shape: {y.shape}")
print(f"  Min: {y.min():.2f} | Max: {y.max():.2f} | Mean: {y.mean():.2f}")

print(f"\nFeature matrix (X):")
print(f"  Shape: {X.shape}")
print(f"  Features: {X.columns.tolist()}")

✓ Features and target defined

Target variable (y): gross income
  Shape: (1000,)
  Min: 0.48 | Max: 874.98 | Mean: 114.79

Feature matrix (X):
  Shape: (1000, 14)
  Features: ['Branch', 'Customer type', 'Gender', 'Product line', 'Unit price', 'Quantity', 'Total', 'Payment', 'Rating', 'hour', 'is_weekend', 'day_num', 'month_num', 'price_quantity']


## 6. Train/Test Split — Temporal Strategy
Splitting data chronologically to prevent data leakage

In [7]:
# ── TEMPORAL TRAIN/TEST SPLIT ─────────────────────────────
# We use 80% for training, 20% for testing
# IMPORTANT: We do NOT shuffle — order must be preserved
# because earlier data should train the model,
# later data should test it (mimics real-world forecasting)

split_index = int(len(df) * 0.8)  # 80% of 1000 = 800

X_train = X.iloc[:split_index]    # rows 0 to 799
X_test  = X.iloc[split_index:]    # rows 800 to 999
y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

print("✓ Train/Test split complete")
print(f"\nTraining set  : {X_train.shape[0]} rows ({X_train.shape[0]/len(df)*100:.0f}%)")
print(f"Test set      : {X_test.shape[0]} rows ({X_test.shape[0]/len(df)*100:.0f}%)")
print(f"\nX_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")
print(f"y_train shape : {y_train.shape}")
print(f"y_test shape  : {y_test.shape}")

✓ Train/Test split complete

Training set  : 800 rows (80%)
Test set      : 200 rows (20%)

X_train shape : (800, 14)
X_test shape  : (200, 14)
y_train shape : (800,)
y_test shape  : (200,)


## 7. Feature Scaling — StandardScaler
Normalising numerical features to the same scale

In [8]:
# ── STANDARD SCALING ─────────────────────────────────────
# StandardScaler transforms features so they have:
# mean = 0 and standard deviation = 1
# This prevents features with large ranges dominating the model

scaler = StandardScaler()

# IMPORTANT: fit ONLY on training data, then transform both
# If we fit on all data, test data "leaks" into training — data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Convert back to DataFrame so column names are preserved
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X_test.columns)

print("✓ Scaling complete")
print(f"\nBefore scaling — Unit price range:")
print(f"  Min: {X_train['Unit price'].min():.2f} | Max: {X_train['Unit price'].max():.2f}")
print(f"\nAfter scaling — Unit price range:")
print(f"  Min: {X_train_scaled['Unit price'].min():.2f} | Max: {X_train_scaled['Unit price'].max():.2f}")
print(f"\nAfter scaling — all means should be ~0:")
print(X_train_scaled.mean().round(3))

✓ Scaling complete

Before scaling — Unit price range:
  Min: 10.13 | Max: 99.96

After scaling — Unit price range:
  Min: -1.72 | Max: 1.67

After scaling — all means should be ~0:
Branch            0.0
Customer type     0.0
Gender            0.0
Product line     -0.0
Unit price       -0.0
Quantity         -0.0
Total            -0.0
Payment           0.0
Rating            0.0
hour             -0.0
is_weekend        0.0
day_num           0.0
month_num        -0.0
price_quantity   -0.0
dtype: float64


## 8. Save Pipeline Objects
Saving encoders and scaler for use in Phase 4, 5, and Flask API

In [9]:
# ── SAVE ALL PIPELINE OBJECTS ────────────────────────────
import os
os.makedirs('../model', exist_ok=True)

# Save the scaler
with open('../model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save all encoders
with open('../model/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

# Save the processed datasets
X_train_scaled.to_csv('../data/X_train.csv', index=False)
X_test_scaled.to_csv('../data/X_test.csv',   index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv',   index=False)

print("✓ Pipeline objects saved:")
print("  ../model/scaler.pkl       — StandardScaler")
print("  ../model/encoders.pkl     — LabelEncoders for all categorical columns")
print("  ../data/X_train.csv       — Training features (scaled)")
print("  ../data/X_test.csv        — Test features (scaled)")
print("  ../data/y_train.csv       — Training target")
print("  ../data/y_test.csv        — Test target")
print(f"\n✓ X_train shape: {X_train_scaled.shape}")
print(f"✓ X_test shape : {X_test_scaled.shape}")
print(f"✓ y_train shape: {y_train.shape}")
print(f"✓ y_test shape : {y_test.shape}")

✓ Pipeline objects saved:
  ../model/scaler.pkl       — StandardScaler
  ../model/encoders.pkl     — LabelEncoders for all categorical columns
  ../data/X_train.csv       — Training features (scaled)
  ../data/X_test.csv        — Test features (scaled)
  ../data/y_train.csv       — Training target
  ../data/y_test.csv        — Test target

✓ X_train shape: (800, 14)
✓ X_test shape : (200, 14)
✓ y_train shape: (800,)
✓ y_test shape : (200,)
